In [1]:
import pandas as pd

## 0. Load Common Metadata And Subset Raw

In [2]:
TYPE_CALIBRATOR = "Calibrator"

samples = pd.read_table(
    "/mnt/data/test/sample.txt"
)
samples.to_parquet(
    "/mnt/code/preprocess-somascan-data/tests/data/samples.parquet.gz",
    compression = "gzip",
    index = False
)

features = pd.read_table(
    "/mnt/data/test/somamer.txt"
).rename({"SeqId": "ProbeId"}, axis = 1)
features.to_parquet(
    "/mnt/code/preprocess-somascan-data/tests/data/features.parquet.gz",
    compression = "gzip",
    index = False
)

In [3]:
# Read in raw data
measurements_raw = pd.read_table(
    "/mnt/data/test/RFU_raw.txt",
    header=None
)
measurements_raw.index = samples.set_index(["PlateId", "PlatePosition"]).index
measurements_raw.columns = features["ProbeId"]

# Melt raw data
measurements_raw = (
    measurements_raw.reset_index()
                         .melt(
                             id_vars=["PlateId", "PlatePosition"],
                             var_name="ProbeId"
                         )
)

# Write expected data
measurements_raw.to_parquet(
    "/mnt/code/preprocess-somascan-data/tests/data/measurements.parquet.gz",
    compression = "gzip",
    index = False
)

## 1. Hybridization control normalization

In [4]:
# Read in expected data
measurements_expected = pd.read_table(
    "/mnt/data/test/RFU_hyb.txt",
    header=None
)
measurements_expected.index = samples.set_index(["PlateId", "PlatePosition"]).index
measurements_expected.columns = features["ProbeId"]

# Melt expected data
measurements_expected = (
    measurements_expected.reset_index()
                         .melt(
                             id_vars=["PlateId", "PlatePosition"],
                             var_name="ProbeId"
                         )
)

# Write expected data
measurements_expected.to_parquet(
    "/mnt/code/preprocess-somascan-data/tests/data/measurements_hcn.parquet.gz",
    compression = "gzip",
    index = False
)

In [5]:
measurements_expected

,PlateId,PlatePosition,ProbeId,value
0,P0031168,A1,10000-28,628.8625
1,P0031168,A10,10000-28,621.3531
2,P0031168,A11,10000-28,550.5609
3,P0031168,A12,10000-28,557.3481
4,P0031168,A2,10000-28,394.5846
...,...,...,...,...
15571795,P0031201,H5,9999-1,2903.1020
15571796,P0031201,H6,9999-1,1622.1810
15571797,P0031201,H7,9999-1,7548.1290
15571798,P0031201,H8,9999-1,6344.4030


In [6]:
# Read in test data
measurements_test = pd.read_csv(
    "/mnt/data/processed/measurements.hybridization_control_normalized.csv"
)

In [7]:
measurements_test

,PlateId,PlatePosition,ProbeId,value
0,P0031168,A1,10000-28,628.862531
1,P0031168,A10,10000-28,621.353074
2,P0031168,A11,10000-28,550.560857
3,P0031168,A12,10000-28,557.348126
4,P0031168,A2,10000-28,394.584597
...,...,...,...,...
15571795,P0031201,H5,9999-1,2903.102446
15571796,P0031201,H6,9999-1,1622.180841
15571797,P0031201,H7,9999-1,7548.129488
15571798,P0031201,H8,9999-1,6344.403187


In [8]:
measurements_joined = measurements_expected.set_index(
    ["PlateId", "PlatePosition", "ProbeId"]
).value.rename("value_expected").to_frame().join(
    measurements_test.set_index(
        ["PlateId", "PlatePosition", "ProbeId"]
    ).value.rename("value_test")
)

In [9]:
measurements_joined["value_diff_ppm"] = 1e6 * (
    measurements_joined.value_expected - measurements_joined.value_test
) / measurements_joined.value_expected

assert measurements_joined.value_diff_ppm.max() < 1

## Check test data folder

In [10]:
ls -lh /mnt/code/preprocess-somascan-data/tests/data/

total 955M
-rw-r--r--. 1 ubuntu ubuntu 362K Aug 31 20:21 features.csv
-rw-r--r--. 1 ubuntu ubuntu 362K Aug 31 20:23 features.parquet.gz
-rw-r--r--. 1 ubuntu ubuntu    0 Aug 31 19:12 __init__.py
-rw-r--r--. 1 ubuntu ubuntu 405M Aug 31 20:15 measurements.csv
-rw-r--r--. 1 ubuntu ubuntu 439M Aug 31 20:19 measurements_hcn.csv
-rw-r--r--. 1 ubuntu ubuntu  70M Aug 31 20:23 measurements_hcn.parquet.gz
-rw-r--r--. 1 ubuntu ubuntu  42M Aug 31 20:23 measurements.parquet.gz
-rw-r--r--. 1 ubuntu ubuntu 140K Aug 31 20:21 samples.csv
-rw-r--r--. 1 ubuntu ubuntu 140K Aug 31 20:23 samples.parquet.gz
